# Tests

In [ ]:
import gc
import os
import sys
from pathlib import Path

os.environ.setdefault("PYTHONDONTWRITEBYTECODE", "1")

try:
    import google.colab
    from google.colab import drive

    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

src_dir = Path("/content/drive/MyDrive/Research/AI-In-Science-Lab/src/local_learning")
if not (src_dir / "datasets").is_dir() or not (src_dir / "models").is_dir():
    raise FileNotFoundError(f"Expected local_learning source directory at {src_dir}.")

os.chdir(src_dir)
sys.path = [path for path in sys.path if path != str(src_dir)]
sys.path.insert(0, str(src_dir))
for module_name in list(sys.modules):
    if module_name == "datasets" or module_name.startswith("datasets."):
        del sys.modules[module_name]
print(f"[src] {src_dir}")

import torch
from datasets import get_dataset
from models import get_model

DATASETS = (
    "cifar10",
    "smallcifar10",
    "cifar100",
    "cifar10_patches",
    "smallcifar10_patches",
    "cifar100_patches",
)

ARCHITECTURES = (
    "cnn1",
    "cnn2",
    "resnet18",
    "vgg16",
    "greedy_stacked_autoencoder",
    "wta_conv_ae",
)

def base_cfg(architecture_type: str) -> dict:
    training_mode = "reconstruction" if architecture_type == "wta_conv_ae" else "classification"
    return {
        "architecture_type": architecture_type,
        "dataset": "cifar10",
        "training_mode": training_mode,
        "epochs": 1,
        "hidden_channels": [8, 8],
        "num_layers": 2,
        "vgg16_first_full_training_epoch": None,
        "vgg16_deconv_training": False,
        "vgg16_local_training": False,
        "vgg16_small": True,
        "k_lifetime": 0.2,
        "k_spatial": None,
        "k_population": None,
        "k_population_alpha": 1.0,
        "gsa_hidden_channels": [8, 8],
        "gsa_local_training": True,
        "gsa_local_reconstruction_strength": 1.0,
    }

dataset_results = []
model_results = []

## Datasets

In [ ]:
for dataset in DATASETS:
    for train in (True, False):
        ds = get_dataset(train=train, dataset=dataset, preprocessing="none")
        image, label = ds[0]
        result = {
            "dataset": dataset,
            "split": "train" if train else "test",
            "length": len(ds),
            "image_shape": tuple(image.shape),
            "label_shape": tuple(torch.as_tensor(label).shape),
        }
        dataset_results.append(result)
        print(result)

        del ds, image, label
        gc.collect()

## Models

In [ ]:
for architecture_type in ARCHITECTURES:
    model = get_model(base_cfg(architecture_type))
    total_params = sum(param.numel() for param in model.parameters())
    trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
    result = {
        "architecture_type": architecture_type,
        "class": type(model).__name__,
        "total_params": total_params,
        "trainable_params": trainable_params,
    }
    assert isinstance(model, torch.nn.Module)
    assert trainable_params > 0
    model_results.append(result)
    print(result)

    del model
    gc.collect()

## Summary

In [ ]:
print(f"Datasets checked: {len(dataset_results)}")
print(f"Models checked: {len(model_results)}")
assert len(dataset_results) == len(DATASETS) * 2
assert len(model_results) == len(ARCHITECTURES)